In [1]:
# import pandas
import pandas as pd
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Page

In [2]:
# import ccxt dataframe
avg_df = pd.read_pickle('files\\avg_df.pkl')

# convert timestamp to datetime
avg_df['month'] = avg_df['timestamp'].dt.month 

In [6]:
# Plot average price for each symbol using pyecharts (dark theme)
# Create responsive chart for embedding
fig = Line(
    init_opts=opts.InitOpts(
        theme="dark",
        bg_color="rgba(0,0,0,0)",
        width="100%",  # Responsive width
        height="100%",  # Responsive height
        page_title="Crypto Price History",
        # Using public CDN for better accessibility
        js_host="https://cdn.jsdelivr.net/npm/echarts@latest/dist/"
    )
)

# Prepare x-axis as unique date strings for plotting
x_axis = avg_df['timestamp'].sort_values().unique()
x_axis_str = [ts.strftime('%Y-%m-%d') for ts in x_axis]
fig.add_xaxis(x_axis_str)

# Add a line for each symbol, hide data labels
for symbol, group in avg_df.groupby('symbols'):
    group = group.set_index('timestamp')
    y_axis = [group['average_price'].get(ts, None) for ts in x_axis]
    fig.add_yaxis(
        symbol, y_axis,
        label_opts=opts.LabelOpts(is_show=False)
    )

# Add responsive rendering options
fig.set_global_opts(
    # Add responsive legend
    
    
    # Ensure tooltip works on mobile
    tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
    
    # Add data zoom slider and set x-axis label formatter
    datazoom_opts=[
        opts.DataZoomOpts(type_="slider", range_start=0, range_end=100),
        opts.DataZoomOpts(type_="inside")
    ],
    yaxis_opts=opts.AxisOpts(type_="log"),
    xaxis_opts=opts.AxisOpts(
        type_="category",
        axislabel_opts=opts.LabelOpts(interval="auto", rotate=30),
        splitline_opts=opts.SplitLineOpts(is_show=False)
    ),
    
    # Add title to the chart
    title_opts=opts.TitleOpts(
        title="Price History (Log Scale)",
        subtitle="Weekly average prices"
    )
)

# Render for embedding with a template that includes viewport meta tag
fig.render('echarts\\embed_chart.html', template_name="simple_chart.html")

fig.render_notebook()  # For Jupyter Notebook display

C:\Users\Matheus\AppData\Local\Temp\ipykernel_38864\1809130306.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for symbol, group in avg_df.groupby('symbols'):
